# Delta-CeNN V5.6 — Four-Way Qwen3.5-0.8B Architecture Recovery

**A vs B vs C vs D** · 18 chunk-cell Delta-CeNN layers · 6 optional V5.3 full-attention layers · validation-gated rollback · native-vs-student generation · automatic logs/checkpoint/metrics download.

This experiment investigates the V5.5 failure pattern: forced 24-layer replacement preserved next-token NLL but lost agreement (~91.8%) and was slower (~0.79×). The new design separates the cost/quality of the 18 linear-attention substitutions from that of the six full-attention substitutions, and retains a usable research candidate only after validation qualification.

**Four arms:** A original Qwen; B all 18 chunk-cell Delta-CeNN plus six native attention layers; C arm B plus only accepted V5.3 full-attention heads; D forced 24-layer *research-only* evaluation. No checkpoint upload; the notebook retrains in this session. The fast Delta-CeNN uses Qwen's chunked gated-delta kernel plus CeNN state mixing, not an independent recurrence. Training and evaluation use text-only unpadded `use_cache=False` inference. No inference speedup, publication claim, or HF publish is implied by completing the notebook.


## 0 · Setup, source provenance, and preflight tests
Select a Google Colab **NVIDIA GPU** and **Runtime → Run all**. The Colab clones GitHub `main`, runs unit and notebook-contract tests, pins the Qwen3.5 model SHA, and writes fully reproducible metadata. No files need to be uploaded.


In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.36,<2" "datasets>=3,<5" "accelerate>=1" "safetensors" "pandas>=2" "matplotlib>=3.7" "pytest>=8"


In [ ]:
import os,sys,subprocess,gc,json,hashlib,math,random,time,csv,zipfile,traceback,shutil
from pathlib import Path
from contextlib import contextmanager,nullcontext
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
REPO=Path('/content/TinyCeNN-LM')
if not (REPO/'.git').is_dir():
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','main',
        'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
sys.path.insert(0,str(REPO/'src'))
from tinycenn_lm.cenn_v56_fourway import (ARMS,select_validation_arm,temporary_full_attention)
from tinycenn_lm.cenn_v53_multilayer import (FULL_ATTENTION_LAYERS,FullAttentionGraftManager,
    validate_full_attention_targets,check_quality,experiment_fingerprint,append_jsonl)
from tinycenn_lm.delta_cenn_qwen35 import Qwen35DeltaCeNNLinear
from tinycenn_lm.delta_cenn_qwen35_v55 import Qwen35DeltaCeNNChunkLinear
from tinycenn_lm.delta_cenn_v55_io import write_adapter_package,prepare_hf_model_card
from tinycenn_lm.cenn_deltacell_v3 import CeNNDeltaCellV3
from tinycenn_lm.cenn_head_rescue_v5 import Qwen35HeadRescueV5
from tinycenn_lm.cenn_head_rescue_v51 import CeNNQueryRescueV51,warmstart_v5_delta_core
from tinycenn_lm.cenn_head_rescue_v52 import CeNNHeadRescueV52,warmstart_v51_to_v52
from tinycenn_lm.cenn_head_rescue_v53 import CeNNHeadRescueV53,warmstart_v52_to_v53
from transformers import AutoConfig,AutoTokenizer,Qwen3_5ForCausalLM
from huggingface_hub import model_info
subprocess.run([sys.executable,'-m','pytest','-q',
    str(REPO/'tests/test_delta_cenn_qwen35.py'),
    str(REPO/'tests/test_cenn_head_rescue_v53.py'),
    str(REPO/'tests/test_qwen35_v53_all24_contract.py'),
    str(REPO/'tests/test_delta_cenn_v54_io.py'),
    str(REPO/'tests/test_delta_cenn_v55_chunk.py'),
    str(REPO/'tests/test_qwen35_v55_notebook.py'),
    str(REPO/'tests/test_cenn_v56_fourway.py'),
    str(REPO/'tests/test_qwen35_v56_notebook.py')],check=True)
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → NVIDIA GPU'
DEVICE=torch.device('cuda')
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
SOURCE_FILES=[REPO/'src/tinycenn_lm'/f for f in ('delta_cenn_qwen35.py',
    'delta_cenn_v55_io.py','delta_cenn_qwen35_v55.py','cenn_v56_fourway.py','cenn_v53_multilayer.py','cenn_deltacell_v3.py','cenn_head_rescue_v5.py',
    'cenn_head_rescue_v51.py','cenn_head_rescue_v52.py','cenn_head_rescue_v53.py')]
SOURCE_SHA256=hashlib.sha256(b''.join(p.read_bytes() for p in SOURCE_FILES)).hexdigest()
print('Git:',GIT_SHA,'model dtype:',DTYPE,'device:',torch.cuda.get_device_name(0))


## 1 · Experiment protocol and controls
Use `quick` for a mechanical sanity check and `research` for longer calibration. **B is always measured with all 18 trained Delta-CeNN modules; C keeps only validated full-attention candidates; D forces all full-attention candidates temporarily.** Selection relies on validation only. Audit is a descriptive internal WikiText-2 train-tail diagnostic, not an independent test. No automatic promotion to deployable status.


In [ ]:
# @title 24-layer Delta-CeNN / CeNN V5.3 research controls
PROFILE='quick' # @param ['quick','research']
FORCE_ALL_24=False  # Deprecated: forced comparisons only occur in arm D, never as final selected model
REPLACEMENT_MODE='one_group' # @param ['one_group','both_groups']
AUTO_DOWNLOAD_RESULTS=True # @param {type:'boolean'}
MODEL_ID='Qwen/Qwen3.5-0.8B'
SEQ_LEN=128 if PROFILE=='quick' else 256
TRAIN_BLOCKS=48 if PROFILE=='quick' else 160
DEV_BLOCKS=24 if PROFILE=='quick' else 64
AUDIT_BLOCKS=16 if PROFILE=='quick' else 48
AUDIT_START_TOKEN=1_850_000
BATCH_SIZE=1 if PROFILE=='quick' else 2
DELTA_STEPS=80 if PROFILE=='quick' else 320
BASELINE_STEPS=60 if PROFILE=='quick' else 240
V53_STEPS=96 if PROFILE=='quick' else 400
DELTA_LR=5e-5
BASELINE_LR=2e-3
V53_LR=6e-4
DELTA_FEATURE_DIM=None  # None = native full-rank key features; compressed < key dimension is experimental
DELTA_CELL_INTERVAL=64   # Two cellular state updates per 128-token sequence
BACKEND_CHUNK_SIZE=64  # Qwen chunked gated-delta primitive; not independent recurrence
SKIP_NEAR_EXACT_FIT=True # Skip training when step-zero already has excellent local parity
FEATURE_DIM=64
LOCAL_WINDOW=64 if PROFILE=='quick' else 128
CHUNK_SIZE=32
RANK=16
SEED=5318
LOG_EVERY=10 if PROFILE=='quick' else 25
MIN_COSINE=.95;MIN_AGREEMENT=.98;MAX_DELTA_NLL=.02;MIN_SPEEDUP=1.05
MIN_RESEARCH_SPEEDUP=.90  # lower bound for accepting B/C as a research model
MAX_FULL_MARGINAL_NLL=.010  # max incremental validation NLL after a full-attention candidate
MIN_FULL_RELATIVE_SPEED=.95  # reject any full candidate >5% slower than current hybrid
EARLY_STOP_PATIENCE=5 if PROFILE=='quick' else 9
EARLY_STOP_TOL=1e-5
RUN_FAST_EVAL=True  # True: real-token NLL, next-token gold accuracy and greedy text probes
MAX_NEW_TOKENS=8 if PROFILE=='quick' else 24
HF_UPLOAD_MODE='disabled'  # @param ['research','quality_only','disabled']
HF_REPO_ID='vtava/Qwen35-08B-DeltaCeNN-V56-FourWay-Research'
ENABLE_COLAB_SECRET_LOOKUP=False  # Avoid HF_TOKEN vault timeout; set env HF_TOKEN or enable explicitly
OUT=Path('/content/qwen35_delta_cenn_v56_fourway_results')
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
TRAIN_EVENTS=OUT/'training_steps.jsonl'
TEXT_LOG=OUT/'training.log'
STAGE_CSV=OUT/'cumulative_layer_results.csv'
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True
print('Full-attention mode',REPLACEMENT_MODE,'four arms',ARMS, 'no forced deployment')
print('Optimizers: Delta',DELTA_STEPS,'full seed',BASELINE_STEPS,'V5.3',V53_STEPS,'HF mode',HF_UPLOAD_MODE)


## 2 · Frozen native Qwen and model-parity controls
The base Qwen has 24 decoder layers: six full-attention and 18 gated-delta. Pretrained projections, convolution, output norms, and MLP stay frozen. The notebook requires exact native-adapter parity before training.


In [ ]:
REVISION=model_info(MODEL_ID).sha
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
assert cfg.model_type=='qwen3_5_text' and cfg.hidden_size==1024 and cfg.num_hidden_layers==24
FULL_TARGETS=tuple(i for i,t in enumerate(cfg.layer_types) if t=='full_attention')
DELTA_TARGETS=tuple(i for i,t in enumerate(cfg.layer_types) if t=='linear_attention')
assert FULL_TARGETS==FULL_ATTENTION_LAYERS and len(DELTA_TARGETS)==18
TARGET_LAYERS=tuple(range(cfg.num_hidden_layers))
assert sorted(FULL_TARGETS+DELTA_TARGETS)==list(TARGET_LAYERS)
TOKENIZER=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,dtype=DTYPE,
    attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False);model.config.use_cache=False
LAYERS=model.model.layers
MANAGER=FullAttentionGraftManager(LAYERS,FULL_TARGETS)
DELTA_NATIVE={i:LAYERS[i].linear_attn for i in DELTA_TARGETS}
INSTALLED_DELTA={}
HD=int(MANAGER.native[11].head_dim)
QHEADS=int(MANAGER.native[11].q_proj.out_features//(2*HD))
KVHEADS=int(MANAGER.native[11].k_proj.out_features//HD)
assert (QHEADS,KVHEADS,HD)==(8,2,256)
EXACT_GROUPS=(0,) if REPLACEMENT_MODE=='one_group' else ()
APPROX_KV=KVHEADS-len(EXACT_GROUPS)
APPROX_Q=APPROX_KV*(QHEADS//KVHEADS)
@contextmanager
def native_everywhere():
    restored={i:LAYERS[i].linear_attn for i in DELTA_TARGETS}
    with MANAGER.native_context():
        try:
            for i,original in DELTA_NATIVE.items():LAYERS[i].linear_attn=original
            yield
        finally:
            for i,old in restored.items():LAYERS[i].linear_attn=old
@contextmanager
def delta_candidate(idx,adapter):
    assert idx in DELTA_NATIVE and adapter.original is DELTA_NATIVE[idx]
    assert LAYERS[idx].linear_attn is DELTA_NATIVE[idx]
    LAYERS[idx].linear_attn=adapter
    try:yield
    finally:LAYERS[idx].linear_attn=DELTA_NATIVE[idx]
def install_delta(idx,adapter):
    assert idx in DELTA_NATIVE and adapter.original is DELTA_NATIVE[idx]
    assert LAYERS[idx].linear_attn is DELTA_NATIVE[idx]
    LAYERS[idx].linear_attn=adapter
    INSTALLED_DELTA[idx]=adapter
def installed_indices():
    return sorted(set(MANAGER.installed_layers)|set(INSTALLED_DELTA))
# Numerical parity controls on a text-only prefix, before training.
probe=TOKENIZER('Delta CeNN compares causal learned memory with original Qwen.',
    add_special_tokens=False,return_tensors='pt').input_ids.to(DEVICE)
with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
    original_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
    oracle=Qwen35HeadRescueV5(MANAGER.native[FULL_TARGETS[0]],None,exact_groups=tuple(range(KVHEADS))).eval()
    with MANAGER.candidate(FULL_TARGETS[0],oracle):
        exact_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
full_parity=float((original_logits-exact_logits).abs().max())
assert full_parity<.05 and bool((original_logits.argmax(-1)==exact_logits.argmax(-1)).all())
linear_idx=DELTA_TARGETS[0]
# Two DISTINCT controls. The native-backend oracle proves that convolution,
# QKV projections, gates, normalization, and output projection are wired
# correctly. The genuine Delta-CeNN tokenwise recurrence is then measured as
# an experimental approximation, not mislabeled a bit-exact chunk kernel.
backend_oracle=Qwen35DeltaCeNNLinear(DELTA_NATIVE[linear_idx],feature_dim=None,
    cells=False,native_backend_oracle=True).to(DEVICE).eval()
token_oracle=Qwen35DeltaCeNNLinear(DELTA_NATIVE[linear_idx],feature_dim=None,
    cells=False).to(DEVICE).eval()
with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
    with delta_candidate(linear_idx,backend_oracle):
        backend_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
    with delta_candidate(linear_idx,token_oracle):
        token_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()

def parity_metrics(reference, candidate):
    a,b=reference.float(),candidate.float()
    return dict(max_abs=float((a-b).abs().max().item()),
        nmse=float(((a-b).square().mean()/a.square().mean().clamp_min(1e-12)).item()),
        cosine=float(F.cosine_similarity(a.reshape(-1,a.shape[-1]),
                   b.reshape(-1,b.shape[-1]),dim=-1).mean().item()),
        top1_agreement=float((a.argmax(-1)==b.argmax(-1)).float().mean().item()))
PARITY=dict(layer=int(linear_idx),backend='Qwen3.5 chunk gated-delta',
    model_revision=REVISION,source_git_sha=GIT_SHA,dtype=str(DTYPE),
    exact_full_attention=parity_metrics(original_logits,exact_logits),
    adapter_backend_oracle=parity_metrics(original_logits,backend_logits),
    experimental_token_delta=parity_metrics(original_logits,token_logits))
(OUT/'parity_diagnostics.json').write_text(json.dumps(PARITY,indent=2,allow_nan=False))
print('PARITY CONTROLS',json.dumps(PARITY,indent=2))
# Keep a STRICT backend parity requirement: if this fails, the projection or
# backend integration is actually wrong. Do NOT relax the threshold based on
# the experimental tokenwise recurrence's much larger absolute logit errors.
assert PARITY['adapter_backend_oracle']['max_abs']<.05, (
    'Native-backend adapter integration failed: inspect parity_diagnostics.json')
assert PARITY['adapter_backend_oracle']['top1_agreement']==1.0, (
    'Native-backend adapter changed teacher top-1; inspect parity diagnostics')
if PARITY['experimental_token_delta']['max_abs']>=.15:
    print('NOTE: tokenwise Delta-CeNN diverges numerically from the chunk backend. '
          'This is measured approximation error, not native-adapter parity failure; '
          'continue only as an experimental student and require strict validation gates.')
print('Pinned model',REVISION,'full layers',FULL_TARGETS,'delta layers',DELTA_TARGETS)
del oracle,backend_oracle,token_oracle,probe,original_logits,exact_logits,backend_logits,token_logits
# Diagnostic output is already saved, even if the subsequent training fails.
torch.cuda.empty_cache()


## 2A · Isolated runtime benchmark
Compare Qwen's native gated-delta path, V5.4's tokenwise Delta-CeNN, and V5.5's chunk-cell engine on the same GPU. This is a single-layer engine check, not proof that a full 24-layer student is faster.


In [ ]:
# @title Single-layer engine comparison on actual Qwen text
@torch.inference_mode()
def benchmark_recurrence_backends():
    sample=TOKENIZER(('Comparing fast recurrent memory updates against the native '
                      'Qwen gated-delta model. ')*9,add_special_tokens=False,
                     return_tensors='pt')['input_ids'][:,:96].to(DEVICE)
    idx=DELTA_TARGETS[0]
    modes={
      'native':None,
      'v54_tokenwise':Qwen35DeltaCeNNLinear(DELTA_NATIVE[idx],feature_dim=None,
                      cells=True,cell_interval=16).to(DEVICE).eval(),
      'v55_chunk_cell':Qwen35DeltaCeNNChunkLinear(DELTA_NATIVE[idx],feature_dim=None,
                      cells=True,cell_interval=DELTA_CELL_INTERVAL,
                      backend_chunk_size=BACKEND_CHUNK_SIZE).to(DEVICE).eval(),
    }
    benchmark={}
    for name,adapter in modes.items():
        readings=[]
        context=(delta_candidate(idx,adapter) if adapter is not None else nullcontext())
        with context:
            for attempt in range(9):
                begin=torch.cuda.Event(enable_timing=True)
                end=torch.cuda.Event(enable_timing=True)
                begin.record()
                with torch.autocast('cuda',dtype=DTYPE):
                    logits=model(input_ids=sample,use_cache=False,
                                 logits_to_keep=1).logits
                end.record();torch.cuda.synchronize()
                assert bool(torch.isfinite(logits).all())
                if attempt>=3:readings.append(begin.elapsed_time(end))
        benchmark[name]=dict(ms_median=float(np.median(readings)),
                             ms_iqr=float(np.percentile(readings,75)-np.percentile(readings,25)))
        print(name,benchmark[name],flush=True)
    benchmark['v55_vs_tokenwise_speedup']=(benchmark['v54_tokenwise']['ms_median']/
                                           benchmark['v55_chunk_cell']['ms_median'])
    benchmark['v55_vs_native_speedup']=(benchmark['native']['ms_median']/
                                        benchmark['v55_chunk_cell']['ms_median'])
    (OUT/'engine_benchmark.json').write_text(json.dumps(benchmark,indent=2))
    return benchmark
ENGINE_BENCHMARK=benchmark_recurrence_backends()


## 3 · Disjoint calibration, validation and diagnostic inputs
Train on early WikiText-2 train windows and rank models on WikiText-2 validation. The train-tail audit is only loaded after final validation selection and must not be treated as independent or untouched scientific validation.


In [ ]:
from datasets import load_dataset
def token_windows(split,count,length,start_token=0):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    needed=start_token+count*length
    ids=[]
    for row in ds:
        sentence=row['text'].strip()
        if not sentence:continue
        ids.extend(TOKENIZER.encode(sentence,add_special_tokens=False))
        ids.append(TOKENIZER.eos_token_id)
        if len(ids)>=needed:break
    if len(ids)<needed:raise RuntimeError(f'{split} has {len(ids)} tokens; need {needed}')
    return torch.tensor(ids[start_token:needed],dtype=torch.long).reshape(count,length)
TRAIN=token_windows('train',TRAIN_BLOCKS,SEQ_LEN)
DEV=token_windows('validation',DEV_BLOCKS,SEQ_LEN)
DATA_HASHES={name:hashlib.sha256(data.numpy().tobytes()).hexdigest()
    for name,data in (('train',TRAIN),('validation',DEV))}
assert TRAIN_BLOCKS*SEQ_LEN<AUDIT_START_TOKEN
CONFIG=dict(model=MODEL_ID,revision=REVISION,repo_sha=GIT_SHA,source_sha256=SOURCE_SHA256,
    profile=PROFILE,experiment='v56_four_way',replacement_mode=REPLACEMENT_MODE,
    all_targets=TARGET_LAYERS,full_targets=FULL_TARGETS,delta_targets=DELTA_TARGETS,
    seq_len=SEQ_LEN,train_blocks=TRAIN_BLOCKS,dev_blocks=DEV_BLOCKS,
    audit_blocks=AUDIT_BLOCKS,audit_offset=AUDIT_START_TOKEN,
    delta_steps=DELTA_STEPS,baseline_steps=BASELINE_STEPS,v53_steps=V53_STEPS,
    delta_feature_dim=DELTA_FEATURE_DIM,delta_cell_interval=DELTA_CELL_INTERVAL,
    batch_size=BATCH_SIZE,local_window=LOCAL_WINDOW,seed=SEED,data_hashes=DATA_HASHES,
    min_agreement=MIN_AGREEMENT,max_delta_nll=MAX_DELTA_NLL,
    min_research_speedup=MIN_RESEARCH_SPEEDUP,
    max_full_marginal_nll=MAX_FULL_MARGINAL_NLL,min_full_relative_speed=MIN_FULL_RELATIVE_SPEED,
    torch_version=torch.__version__,gpu=torch.cuda.get_device_name(0))
RUN_FINGERPRINT=experiment_fingerprint(CONFIG)
(OUT/'config.json').write_text(json.dumps(CONFIG,indent=2,allow_nan=False))
print('Train',TRAIN.shape,'Validation',DEV.shape,'fingerprint',RUN_FINGERPRINT[:16])


## 4 · Reproducible layer capture, loss, and timing
Native teacher module outputs are captured on the upstream representations produced by already installed layers. Delta and full-attention adapters are always trained against the correct native target type; each optimizer event is recorded in `training_steps.jsonl`.


In [ ]:
def log_line(message):
    line=f'[{time.strftime("%Y-%m-%dT%H:%M:%S")}] {message}'
    print(line,flush=True)
    with TEXT_LOG.open('a',encoding='utf-8') as f:f.write(line+'\n')
@torch.inference_mode()
def inference(blocks,keep=1):
    with torch.autocast('cuda',dtype=DTYPE):
        return model(input_ids=blocks.to(DEVICE),use_cache=False,logits_to_keep=keep)
@torch.inference_mode()
def capture_target(idx,split,blocks):
    full=idx in FULL_TARGETS
    native=MANAGER.native[idx] if full else DELTA_NATIVE[idx]
    assert (LAYERS[idx].self_attn if full else LAYERS[idx].linear_attn) is native
    stored={k:[] for k in (('x','y','cos','sin') if full else ('x','y'))}
    def hook(mod,args,kwargs,out):
        x=kwargs.get('hidden_states')
        if x is None and args:x=args[0]
        y=out[0] if isinstance(out,tuple) else out
        stored['x'].append(x.detach().to('cpu',dtype=torch.float16))
        stored['y'].append(y.detach().to('cpu',dtype=torch.float16))
        if full:
            cos,sin=kwargs['position_embeddings']
            stored['cos'].append(cos.detach().cpu().float())
            stored['sin'].append(sin.detach().cpu().float())
    handle=native.register_forward_hook(hook,with_kwargs=True)
    try:
        for start in range(0,len(blocks),BATCH_SIZE):inference(blocks[start:start+BATCH_SIZE])
    finally:handle.remove()
    answer={key:torch.cat(values,0).contiguous() for key,values in stored.items()}
    assert answer['x'].shape[:2]==blocks.shape
    return answer
def captured_batch(data,indices):
    x=data['x'][indices].to(DEVICE,dtype=DTYPE)
    y=data['y'][indices].to(DEVICE,dtype=DTYPE)
    rope=(data['cos'][indices].to(DEVICE),data['sin'][indices].to(DEVICE)) if 'cos' in data else None
    return x,y,rope
def forward_adapter(adapter,x,rope):
    if isinstance(adapter,Qwen35DeltaCeNNLinear):return adapter(x)
    output,_=adapter(x,position_embeddings=rope)
    return output
def mixer_loss(pred,teacher):
    p=pred.float();t=teacher.float()
    nmse=(p-t).square().mean()/t.square().mean().clamp_min(1e-8)
    return nmse+.2*(1-F.cosine_similarity(p,t,dim=-1).mean())
@torch.inference_mode()
def local_validation(adapter,data):
    adapter.eval();err=energy=cosine_sum=0.;n=0
    for lo in range(0,len(data['x']),BATCH_SIZE):
        indices=torch.arange(lo,min(lo+BATCH_SIZE,len(data['x'])))
        x,y,rope=captured_batch(data,indices)
        with torch.autocast('cuda',dtype=DTYPE):pred=forward_adapter(adapter,x,rope)
        p,t=pred.float(),y.float()
        err+=(p-t).square().sum().item();energy+=t.square().sum().item()
        cosine_sum+=F.cosine_similarity(p,t,dim=-1).sum().item()
        n+=p.shape[0]*p.shape[1]
    return dict(nmse=err/max(energy,1e-8),cosine=cosine_sum/max(n,1))
@torch.inference_mode()
def model_metrics(blocks,teacher_top1=None):
    n=0;nll=0.;correct=0;agreement=0;tops=[]
    for lo in range(0,len(blocks),BATCH_SIZE):
        block=blocks[lo:lo+BATCH_SIZE]
        positions=torch.arange(block.shape[1]-1,device=DEVICE)
        logits=inference(block,keep=positions).logits.float()
        gold=block[:,1:].to(DEVICE)
        logp=F.log_softmax(logits,dim=-1)
        nll-=logp.gather(-1,gold.unsqueeze(-1)).sum().item()
        pred=logits.argmax(-1)
        correct+=(pred==gold).sum().item();n+=gold.numel()
        if teacher_top1 is None:tops.append(pred.cpu().to(torch.int32))
        else:
            reference=teacher_top1[lo//BATCH_SIZE].to(DEVICE)
            assert reference.shape==pred.shape
            agreement+=(reference==pred).sum().item()
    out=dict(nll=nll/n,accuracy=correct/n,decisions=n)
    if teacher_top1 is not None:out['teacher_agreement']=agreement/n
    return out,tops
@torch.inference_mode()
def paired_prefill(blocks,warmups=2,repeats=6):
    sample=blocks[:min(BATCH_SIZE,len(blocks))].to(DEVICE)
    native_times=[];student_times=[]
    def one(native):
        with native_everywhere() if native else nullcontext():
            begin=torch.cuda.Event(enable_timing=True);end=torch.cuda.Event(enable_timing=True)
            begin.record();out=inference(sample,keep=1).logits
            end.record();torch.cuda.synchronize()
            assert bool(torch.isfinite(out).all())
            return begin.elapsed_time(end)
    for i in range(warmups+repeats):
        if i%2:student=one(False);native=one(True)
        else:native=one(True);student=one(False)
        if i>=warmups:native_times.append(native);student_times.append(student)
    return dict(native_ms=float(np.median(native_times)),
        student_ms=float(np.median(student_times)),
        speedup=float(np.median(native_times)/np.median(student_times)),
        native_iqr_ms=float(np.percentile(native_times,75)-np.percentile(native_times,25)),
        student_iqr_ms=float(np.percentile(student_times,75)-np.percentile(student_times,25)),
        repeats=repeats)


## 5 · Building the two separate adapter families
**18 linear layers:** Qwen native chunked gated-delta recurrence plus CeNN state-cell updates every 64 tokens. **6 full-attention layers:** V5.3 selective KV-group rescue using a delta+local seed, then low-rank cross calibration. Training all 24 does not mean installing all 24 in the selected model.


In [ ]:
def make_base(idx):
    core=CeNNDeltaCellV3(APPROX_Q,APPROX_KV,HD,
        feature_dim=FEATURE_DIM,chunk_size=CHUNK_SIZE,mode='none',
        local_window=LOCAL_WINDOW).to(DEVICE)
    return Qwen35HeadRescueV5(MANAGER.native[idx],core,exact_groups=EXACT_GROUPS).eval()
def make_v53(idx,base_state):
    common=dict(num_heads=APPROX_Q,num_kv_heads=APPROX_KV,head_dim=HD,
        feature_dim=FEATURE_DIM,chunk_size=CHUNK_SIZE,local_window=LOCAL_WINDOW,
        correction_rank=RANK,query_router=True,query_correction=True)
    v51=CeNNQueryRescueV51(**common);warmstart_v5_delta_core(v51,base_state)
    v52=CeNNHeadRescueV52(**common,post_rank=RANK,post_correction=True)
    warmstart_v51_to_v52(v52,v51.state_dict())
    v53=CeNNHeadRescueV53(**common,post_rank=RANK,post_correction=True,
        cross_rank=RANK,cross_correction=True,output_calibration=True)
    warmstart_v52_to_v53(v53,v52.state_dict())
    return Qwen35HeadRescueV5(MANAGER.native[idx],v53.to(DEVICE),exact_groups=EXACT_GROUPS).eval()
def make_delta(idx):
    return Qwen35DeltaCeNNChunkLinear(DELTA_NATIVE[idx],feature_dim=DELTA_FEATURE_DIM,
        cells=True,cell_interval=DELTA_CELL_INTERVAL,
        backend_chunk_size=BACKEND_CHUNK_SIZE).to(DEVICE).eval()
def fit_local(idx,stage,adapter,train,dev,steps,lr):
    """Best-of-start + train with validation-based patience; preserve source checkpoint."""
    core=adapter.core
    core.requires_grad_(True)
    optimizer=torch.optim.AdamW(core.parameters(),lr=lr,weight_decay=.005)
    schedule=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=steps,eta_min=lr*.1)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
    generator=torch.Generator().manual_seed(SEED+idx*29+(0 if stage=='delta_cenn' else 5000))
    indices=torch.randint(0,len(train['x']),(steps,BATCH_SIZE),generator=generator)
    initial=local_validation(adapter,dev)
    best=initial['nmse']+.2*(1-initial['cosine'])
    best_local=initial
    best_sd={k:v.detach().cpu().clone() for k,v in core.state_dict().items()}
    best_step=0;stale=0;performed=0;began=time.perf_counter()
    append_jsonl(TRAIN_EVENTS,dict(layer=idx,stage=stage,step=0,
        val_nmse=initial['nmse'],val_cosine=initial['cosine'],
        note='frozen step-zero candidate before learning'))
    log_line(f'L{idx} {stage} init cosine={initial["cosine"]:.6f} NMSE={initial["nmse"]:.6f}')
    for step in range(1,steps+1):
        ix=indices[step-1];x,y,rope=captured_batch(train,ix)
        adapter.train();optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred=forward_adapter(adapter,x,rope)
            loss=mixer_loss(pred,y)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f'L{idx}/{stage} step {step}: non-finite loss')
        scaler.scale(loss).backward();scaler.unscale_(optimizer)
        grad=torch.nn.utils.clip_grad_norm_(core.parameters(),.5)
        scaler.step(optimizer);scaler.update();schedule.step()
        performed=step
        event=dict(layer=idx,family='linear_attention' if idx in DELTA_TARGETS else 'full_attention',
            stage=stage,step=step,total_steps=steps,loss=float(loss.detach()),
            grad_norm=float(grad),lr=float(optimizer.param_groups[0]['lr']),
            elapsed_seconds=float(time.perf_counter()-began))
        if step==1 or step==steps or step%LOG_EVERY==0:
            stats=local_validation(adapter,dev)
            score=stats['nmse']+.2*(1-stats['cosine'])
            if score+EARLY_STOP_TOL<best:
                best=score;best_local=stats;best_step=step;stale=0
                best_sd={k:v.detach().cpu().clone() for k,v in core.state_dict().items()}
            else:
                stale+=1
            event.update(val_cosine=stats['cosine'],val_nmse=stats['nmse'],
                         best_step=best_step,best_score=best)
            log_line(f'L{idx} {stage} {step}/{steps} cos={stats["cosine"]:.5f} '
                f'best={best_local["cosine"]:.5f} step={best_step}')
            if stale>=EARLY_STOP_PATIENCE:
                event['early_stop']=True
                append_jsonl(TRAIN_EVENTS,event)
                log_line(f'L{idx} {stage}: early stopping step {step}; best step {best_step}')
                break
        append_jsonl(TRAIN_EVENTS,event)
    core.load_state_dict(best_sd,strict=True);core.requires_grad_(False);adapter.eval()
    STAGE_TRAINING[(idx,stage)]=dict(requested_steps=steps,performed_steps=performed,
        best_step=best_step,early_stopped=performed<steps,initial_validation=initial,
        best_validation=best_local)
    return best_sd,best_local
@torch.inference_mode()
def warmstart_parity(base,updated,dev):
    x,_,rope=captured_batch(dev,torch.tensor([0]))
    with torch.autocast('cuda',dtype=DTYPE):
        a=forward_adapter(base,x,rope);b=forward_adapter(updated,x,rope)
    diff=float((a-b).abs().max())
    assert diff<.01,diff
    return diff


## 6 · Four separate evaluation arms with progressive training
Run **A** baseline first. Train and install all 18 chunk-cell modules to establish **B** with native full-attention layers. Then train six full-attention candidates one at a time. **C** installs a candidate only if cumulative quality and incremental performance remain within budget. **D** evaluates all 24 replacements temporarily, with guaranteed restoration to C. Select on validation only; a failing B/C falls back to native A. A full training and candidate-checkpoint archive is automatically downloaded even if fitting fails partway.


In [ ]:
# @title V5.6: train 18 Delta-CeNN + six V5.3 candidates; compare A/B/C/D
from tinycenn_lm.cenn_v56_fourway import validate_arm_rows
from contextlib import contextmanager

STAGE_TRAINING={}
REPORT=[]; ARM_VALIDATION=[]; ARM_AUDIT=[]; FULL_CANDIDATES={}
RUN_OK=False; RUN_ERROR=None; RUN_START=time.time()
NATIVE_DEV=None; FINAL=None; ARCHIVE=None; SELECTED_ARM=None

@contextmanager
def arm_context(name):
    """All arms see identical frozen weights and the same validation tokens."""
    if name == 'A_native':
        with native_everywhere():
            yield
    elif name == 'B_delta18':
        with MANAGER.native_context():
            yield  # native full-attention, Delta-CeNN installed in all 18 linear layers
    elif name == 'C_gated_hybrid':
        yield  # installed Delta-CeNN + quality-accepted V5.3 candidates
    elif name == 'D_forced24':
        with temporary_full_attention(LAYERS,MANAGER.native,FULL_CANDIDATES):
            yield  # D is ALWAYS restored; never selected for deployment
    else:
        raise ValueError('Unknown arm '+str(name))

@torch.inference_mode()
def fourway_measure(name, blocks, teacher_top1, native_nll, repeats):
    with arm_context(name):
        if name == 'A_native':
            metrics,_=model_metrics(blocks)
        else:
            metrics,_=model_metrics(blocks,teacher_top1=teacher_top1)
        # Timing's native comparator temporarily restores the entire original backbone.
        timing=paired_prefill(blocks,warmups=2,repeats=repeats)
    return dict(arm=name,nll=metrics['nll'],accuracy=metrics['accuracy'],
        agreement=1.0 if name=='A_native' else metrics['teacher_agreement'],
        delta_nll=metrics['nll']-native_nll,speedup=timing['speedup'],
        native_ms=timing['native_ms'],student_ms=timing['student_ms'],
        installed_linear_layers=(0 if name=='A_native' else len(INSTALLED_DELTA)),
        installed_full_layers=(0 if name in ('A_native','B_delta18')
            else len(MANAGER.installed_layers) if name=='C_gated_hybrid'
            else len(FULL_CANDIDATES)))

try:
    log_line('START V5.6 four-way experiment '+RUN_FINGERPRINT[:16])
    with native_everywhere():
        NATIVE_DEV,NATIVE_TOP1=model_metrics(DEV)
    log_line('A native validation '+json.dumps(NATIVE_DEV))

    # STAGE B: all 18 chunk-cell adapters are trained on progressive inputs.
    # Their acceptance is recorded, but B is deliberately complete and separately
    # benchmarked; this never forces an unsafe selected or released model.
    for layer in DELTA_TARGETS:
        log_line(f'BEGIN Delta-CeNN layer {layer}; progressive prefix {installed_indices()}')
        train_capture=capture_target(layer,'train',TRAIN)
        dev_capture=capture_target(layer,'dev',DEV)
        torch.manual_seed(SEED+layer*13)
        adapter=make_delta(layer)
        state,local=fit_local(layer,'delta_cenn',adapter,
            train_capture,dev_capture,DELTA_STEPS,DELTA_LR)
        checkpoint=OUT/'checkpoints'/f'L{layer:02d}_linear_attention_core.pt'
        torch.save(dict(core_state=state,layer=layer,family='linear_attention',
            core_config=adapter.core.config,validation=local,
            fingerprint=RUN_FINGERPRINT,model_revision=REVISION,source_git_sha=GIT_SHA),checkpoint)
        with delta_candidate(layer,adapter):
            dev,_=model_metrics(DEV,teacher_top1=NATIVE_TOP1)
            timing=paired_prefill(DEV,warmups=2,repeats=4 if PROFILE=='quick' else 10)
        nll_change=dev['nll']-NATIVE_DEV['nll']
        quality=check_quality(local['cosine'],dev['teacher_agreement'],nll_change,
            timing['speedup'],min_cosine=MIN_COSINE,min_agreement=MIN_AGREEMENT,
            max_delta_nll=MAX_DELTA_NLL,min_speedup=MIN_SPEEDUP)
        install_delta(layer,adapter)  # B is a diagnostic arm, not deployment approval
        row=dict(layer=layer,family='linear_attention',arm='B_delta18',
            trained=True,installed_in_B=True,quality_pass=quality.quality_pass,
            cosine=local['cosine'],nmse=local['nmse'],
            dev_agreement=dev['teacher_agreement'],dev_delta_nll=nll_change,
            speedup=timing['speedup'],training=STAGE_TRAINING[(layer,'delta_cenn')])
        REPORT.append(row);pd.DataFrame(REPORT).to_csv(STAGE_CSV,index=False)
        log_line('DELTA SUMMARY '+json.dumps(row,allow_nan=False))
        del train_capture,dev_capture,state
        gc.collect();torch.cuda.empty_cache()

    assert len(INSTALLED_DELTA)==18
    B_VALID=fourway_measure('B_delta18',DEV,NATIVE_TOP1,NATIVE_DEV['nll'],6)
    log_line('B 18-Delta validation '+json.dumps(B_VALID))

    # STAGE C: the native full-attention block is the control for each
    # candidate, and all candidate weights are saved even if rejected.
    previous_metrics=B_VALID
    for layer in FULL_TARGETS:
        log_line(f'BEGIN full-attention layer {layer}; accepted {MANAGER.installed_layers}')
        train_capture=capture_target(layer,'train',TRAIN)
        dev_capture=capture_target(layer,'dev',DEV)
        torch.manual_seed(SEED+layer*13)
        baseline=make_base(layer)
        base_state,base_local=fit_local(layer,'full_delta_seed',baseline,
            train_capture,dev_capture,BASELINE_STEPS,BASELINE_LR)
        torch.save(dict(core_state=base_state,layer=layer,stage='full_delta_seed',
            fingerprint=RUN_FINGERPRINT),OUT/'checkpoints'/f'L{layer:02d}_full_seed.pt')
        adapter=make_v53(layer,base_state)
        warmstart_diff=warmstart_parity(baseline,adapter,dev_capture)
        del baseline;torch.cuda.empty_cache()
        state,local=fit_local(layer,'full_v53',adapter,
            train_capture,dev_capture,V53_STEPS,V53_LR)
        torch.save(dict(core_state=state,layer=layer,family='full_attention',
            core_config=adapter.core.config,validation=local,
            fingerprint=RUN_FINGERPRINT,model_revision=REVISION,
            source_git_sha=GIT_SHA,replacement_mode=REPLACEMENT_MODE),
            OUT/'checkpoints'/f'L{layer:02d}_full_attention_core.pt')
        FULL_CANDIDATES[layer]=adapter
        with MANAGER.candidate(layer,adapter):
            dev,_=model_metrics(DEV,teacher_top1=NATIVE_TOP1)
            timing=paired_prefill(DEV,warmups=2,repeats=4 if PROFILE=='quick' else 10)
        nll_change=dev['nll']-NATIVE_DEV['nll']
        quality=check_quality(local['cosine'],dev['teacher_agreement'],nll_change,
            timing['speedup'],min_cosine=MIN_COSINE,min_agreement=MIN_AGREEMENT,
            max_delta_nll=MAX_DELTA_NLL,min_speedup=MIN_SPEEDUP)
        marginal=dev['nll']-previous_metrics['nll']
        relative_speed=timing['speedup']/max(previous_metrics['speedup'],1e-9)
        accepted=bool(quality.quality_pass and marginal<=MAX_FULL_MARGINAL_NLL
                      and relative_speed>=MIN_FULL_RELATIVE_SPEED)
        if accepted:
            MANAGER.install(layer,adapter)
            previous_metrics=dict(nll=dev['nll'],speedup=timing['speedup'])
        row=dict(layer=layer,family='full_attention',arm='C_gated_hybrid',
            trained=True,accepted_in_C=accepted,quality_pass=quality.quality_pass,
            cosine=local['cosine'],nmse=local['nmse'],
            dev_agreement=dev['teacher_agreement'],dev_delta_nll=nll_change,
            relative_speed=relative_speed,marginal_nll=marginal,
            speedup=timing['speedup'],warmstart_diff=warmstart_diff,
            training=dict(seed=STAGE_TRAINING[(layer,'full_delta_seed')],
                          final=STAGE_TRAINING[(layer,'full_v53')]))
        REPORT.append(row);pd.DataFrame(REPORT).to_csv(STAGE_CSV,index=False)
        log_line('FULL ATTENTION '+('ACCEPT' if accepted else 'ROLLBACK')+' '+json.dumps(row,allow_nan=False))
        del train_capture,dev_capture,state,base_state
        gc.collect();torch.cuda.empty_cache()

    assert set(FULL_CANDIDATES)==set(FULL_TARGETS)
    # All four arms compared *without* changing which candidates C accepted.
    for arm in ARMS:
        record=fourway_measure(arm,DEV,NATIVE_TOP1,NATIVE_DEV['nll'],
                               6 if PROFILE=='quick' else 18)
        ARM_VALIDATION.append(record)
        log_line('FOUR-WAY VALIDATION '+json.dumps(record,allow_nan=False))
    validate_arm_rows(ARM_VALIDATION)
    pd.DataFrame(ARM_VALIDATION).to_csv(OUT/'four_way_validation.csv',index=False)
    SELECTION=select_validation_arm(ARM_VALIDATION,
        min_agreement=MIN_AGREEMENT,max_delta_nll=MAX_DELTA_NLL,
        min_speedup=MIN_RESEARCH_SPEEDUP)
    SELECTED_ARM=SELECTION['selected']
    (OUT/'validation_selection.json').write_text(json.dumps(SELECTION,indent=2))
    log_line('VALIDATION SELECTION '+json.dumps(SELECTION))

    # Audit is NEVER used to select between models. Compare all four only to
    # show the honest consequences of forcing six failed full-attention heads.
    AUDIT=token_windows('train',AUDIT_BLOCKS,SEQ_LEN,start_token=AUDIT_START_TOKEN)
    DATA_HASHES['diagnostic_train_tail']=hashlib.sha256(AUDIT.numpy().tobytes()).hexdigest()
    assert len(set(DATA_HASHES.values()))==3
    with native_everywhere():
        NATIVE_AUDIT,AUDIT_TOP1=model_metrics(AUDIT)
    for arm in ARMS:
        row=fourway_measure(arm,AUDIT,AUDIT_TOP1,NATIVE_AUDIT['nll'],
                            6 if PROFILE=='quick' else 18)
        ARM_AUDIT.append(row)
        log_line('FOUR-WAY DIAGNOSTIC '+json.dumps(row,allow_nan=False))
    pd.DataFrame(ARM_AUDIT).to_csv(OUT/'four_way_diagnostic_audit.csv',index=False)
    selected_audit=next(row for row in ARM_AUDIT if row['arm']==SELECTED_ARM)
    DEPLOY_PASS=bool(SELECTED_ARM in ('B_delta18','C_gated_hybrid')
         and selected_audit['agreement']>=MIN_AGREEMENT
         and selected_audit['delta_nll']<=MAX_DELTA_NLL
         and selected_audit['speedup']>MIN_SPEEDUP)
    FINAL=dict(architecture='V5.6: A native, B Delta18, C validation-gated hybrid, D forced24',
        model=MODEL_ID,revision=REVISION,source_git_sha=GIT_SHA,
        source_sha256=SOURCE_SHA256,run_fingerprint=RUN_FINGERPRINT,
        full_attention_targets=list(FULL_TARGETS),delta_targets=list(DELTA_TARGETS),
        acceptance=dict(min_agreement=MIN_AGREEMENT,max_delta_nll=MAX_DELTA_NLL,
                        max_marginal_full_nll=MAX_FULL_MARGINAL_NLL,
                        min_relative_full_speed=MIN_FULL_RELATIVE_SPEED,
                        min_research_speedup=MIN_RESEARCH_SPEEDUP,
                        min_deploy_speedup=MIN_SPEEDUP),
        validation_selection=SELECTION,validation=ARM_VALIDATION,
        diagnostic_audit=ARM_AUDIT,installed_delta_count=len(INSTALLED_DELTA),
        accepted_full_layers=list(MANAGER.installed_layers),
        forced_D_never_selected=True,final_pass=DEPLOY_PASS,
        data_hashes=DATA_HASHES,run_seconds=time.time()-RUN_START,
        limitations='V5.6 chunked gated-delta primitive is reused from Qwen, with CeNN cell mixing; '
                    'only text-only no-cache prefill; no hybrid KV cache or independent external test.')
    (OUT/'final_audit.json').write_text(json.dumps(FINAL,indent=2,allow_nan=False))
    log_line('FINAL '+json.dumps(dict(selected=SELECTED_ARM,deploy_pass=DEPLOY_PASS,
          selected_audit=selected_audit),allow_nan=False))

    # The live model must match the selected, validation-qualified architecture.
    if SELECTED_ARM=='B_delta18':
        MANAGER.restore_all()
    elif SELECTED_ARM=='A_native':
        MANAGER.restore_all()
        for i,original in DELTA_NATIVE.items():LAYERS[i].linear_attn=original
        INSTALLED_DELTA.clear()
    elif SELECTED_ARM!='C_gated_hybrid':
        raise RuntimeError('Forced D cannot be selected for release')
    assert SELECTED_ARM!='D_forced24'
    RUN_OK=True
except BaseException as exc:
    RUN_ERROR=repr(exc)
    log_line('EXPERIMENT FAILED, saving partial results: '+RUN_ERROR)
    with (OUT/'error_traceback.txt').open('w') as file:traceback.print_exc(file=file)
    raise
finally:
    pd.DataFrame(REPORT).to_csv(STAGE_CSV,index=False)
    status=dict(completed=RUN_OK,error=RUN_ERROR,selected_arm=SELECTED_ARM,
        installed=installed_indices(),
        forced_D_never_deployed=True,run_fingerprint=RUN_FINGERPRINT,
        runtime_seconds=time.time()-RUN_START)
    (OUT/'run_status.json').write_text(json.dumps(status,indent=2,allow_nan=False))
    manifest=[]
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and path.name!='artifact_manifest.json':
            manifest.append(dict(path=str(path.relative_to(OUT)),bytes=path.stat().st_size,
                sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    (OUT/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2))
    archive_path=OUT.parent/(OUT.name+'.zip')
    with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as archive:
        for path in sorted(OUT.rglob('*')):
            if path.is_file():archive.write(path,arcname=str(path.relative_to(OUT)))
    ARCHIVE=str(archive_path)
    print('PARTIAL OR FINAL ARCHIVE',ARCHIVE,'size',archive_path.stat().st_size,flush=True)
    if AUTO_DOWNLOAD_RESULTS and not RUN_OK:
        try:
            from google.colab import files
            files.download(ARCHIVE)
        except Exception as exc:
            print('Download blocked; retrieve ZIP from Colab Files:',ARCHIVE,repr(exc))


## 7 · Fast real-world text sanity check on the **selected** model
Run native-versus-selected Qwen next-token NLL, top-1 decisions, and short greedy chat outputs. Arm D is never substituted into the selected model. Since cached generation is unsupported, use `use_cache=False` and avoid interpreting these few prompts as broad accuracy evidence.


In [ ]:
# @title Fast native-versus-DeltaCeNN end-to-end evaluation
FAST_EVAL_RESULT={}
if RUN_FAST_EVAL:
    try:
        eval_offset=DEV_BLOCKS*SEQ_LEN
        EVAL=token_windows('validation',4,SEQ_LEN,start_token=eval_offset)
        with native_everywhere():
            native_metrics,native_top1=model_metrics(EVAL)
        student_metrics,_=model_metrics(EVAL,teacher_top1=native_top1)
        FAST_EVAL_RESULT=dict(selected_arm=SELECTED_ARM,native=native_metrics,student=student_metrics,
            delta_nll=student_metrics['nll']-native_metrics['nll'],
            installed_layers=installed_indices(),
            evaluation_source='WikiText-2 validation later windows; descriptive, not a blind external test')
        # Chat-template aware greedy comparison; no hybrid cache supported.
        FAST_CHAT_PROMPTS=[
            'What is 12 plus 19? Answer with the number.',
            'What is the capital of France? Answer in one sentence.',
        ]
        def slow_greedy(prompt,max_tokens=8):
            messages=[{'role':'user','content':prompt}]
            try:
                encoded=TOKENIZER.apply_chat_template(messages,tokenize=True,
                    add_generation_prompt=True,return_tensors='pt')
                # HF tokenizers may return Tensor OR BatchEncoding; both valid.
                ids=encoded['input_ids'] if isinstance(encoded,dict) else (
                    encoded.input_ids if hasattr(encoded,'input_ids') else encoded)
            except (TypeError,ValueError,KeyError,AttributeError):
                ids=TOKENIZER('User: '+prompt+'\nAssistant:',return_tensors='pt')['input_ids']
            if ids.ndim==1:ids=ids.unsqueeze(0)
            ids=ids.to(DEVICE)
            start=int(ids.shape[1]);ends=TOKENIZER.eos_token_id
            ends=set(ends if isinstance(ends,(list,tuple)) else [ends])
            im_end=TOKENIZER.convert_tokens_to_ids('<|im_end|>')
            if isinstance(im_end,int) and im_end>=0:ends.add(im_end)
            for _ in range(max_tokens):
                with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
                    logits=model(input_ids=ids,use_cache=False,logits_to_keep=1).logits[:,-1]
                token=logits.argmax(-1,keepdim=True)
                ids=torch.cat((ids,token),dim=-1)
                if int(token.item()) in ends:break
            return TOKENIZER.decode(ids[0,start:].detach().cpu(),skip_special_tokens=True)
        generations=[]
        for question in FAST_CHAT_PROMPTS:
            with native_everywhere(): teacher_text=slow_greedy(question,MAX_NEW_TOKENS)
            student_text=slow_greedy(question,MAX_NEW_TOKENS)
            result=dict(prompt=question,native=teacher_text,student=student_text,
                exact_text_agreement=(teacher_text.strip()==student_text.strip()))
            generations.append(result)
            print('\nPROMPT:',question,'\nNATIVE:',teacher_text,'\nSTUDENT:',student_text,flush=True)
        FAST_EVAL_RESULT['greedy_texts']=generations
        print('FAST GOLD EVAL:',json.dumps({k:v for k,v in FAST_EVAL_RESULT.items()
            if k!='greedy_texts'},indent=2))
    except BaseException as e:
        FAST_EVAL_RESULT={'error':repr(e),'installed_layers':installed_indices()}
        (OUT/'fast_eval_traceback.txt').write_text(traceback.format_exc())
        print('FAST EVAL FAILED:',repr(e))
    finally:
        (OUT/'fast_eval.json').write_text(json.dumps(FAST_EVAL_RESULT,indent=2,allow_nan=False))
else:
    print('Fast eval disabled by setting. Set RUN_FAST_EVAL=True to measure actual language modeling.')


## 8 · Export only the **selected** adapter stack
Package the selected B/C adapter layers, or native A if neither passes validation; all trained rejected and forced-D candidates remain in the research checkpoint archive. The custom adapter is not a standard `AutoModel.from_pretrained` weight snapshot. The original pinned Qwen base is required.


In [ ]:
# @title Export reloadable adapter package and Hugging Face model card
HF_BUNDLE=OUT/'hf_adapter_bundle'
SELECTED=installed_indices()
manifest=write_adapter_package(HF_BUNDLE,OUT,model_id=MODEL_ID,
    model_revision=REVISION,git_sha=GIT_SHA,
    selected_layers=SELECTED,full_layers=FULL_TARGETS,
    replacement_mode=REPLACEMENT_MODE,source_sha256=SOURCE_SHA256,
    run_fingerprint=RUN_FINGERPRINT,final_audit=FINAL,fast_eval=FAST_EVAL_RESULT,
    experimental=not bool(FINAL and FINAL.get('final_pass',False)))
card=prepare_hf_model_card(manifest,HF_REPO_ID)
card=card.replace('Delta-CeNN V5.5 Qwen3.5 research adapter',
    'Delta-CeNN V5.6 four-way Qwen3.5 research adapter')
(HF_BUNDLE/'README.md').write_text(card,encoding='utf-8')
print('Prepared reloadable adapter package:',HF_BUNDLE,'selected',SELECTED,
    'research_only',manifest['research_only'])
(HF_BUNDLE/'four_way_selection.json').write_text(json.dumps(dict(selected_arm=SELECTED_ARM,
  selection=SELECTION,validation=ARM_VALIDATION,diagnostic=ARM_AUDIT),indent=2))


## 9 · Optional Hugging Face upload (disabled by default)
Only upload when a Colab runtime has its own HF write token. The default is `disabled`, and even with `quality_only` the experiment cannot promote forced arm D. Passing validation gates without an audit speed advantage produces a research-only package, not a deployable model.


In [ ]:
# @title Publish adapters to Hugging Face when authorized
HF_STATUS=dict(mode=HF_UPLOAD_MODE,repo_id=HF_REPO_ID,uploaded=False)
try:
    if HF_UPLOAD_MODE not in ('disabled','quality_only','research'):
        raise ValueError('Unknown HF upload policy')
    allowed=HF_UPLOAD_MODE=='research' or (HF_UPLOAD_MODE=='quality_only' and
        bool(FINAL and FINAL.get('final_pass')) and len(SELECTED)>0)
    if HF_UPLOAD_MODE=='disabled':
        HF_STATUS['reason']='Disabled by experiment settings'
    elif not allowed:
        HF_STATUS['reason']='No validated deployable model: gated upload skipped'
    else:
        token=os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
        if not token and ENABLE_COLAB_SECRET_LOOKUP:
            try:
                from google.colab import userdata
                token=userdata.get('HF_TOKEN')
            except Exception as e:
                HF_STATUS['secret_lookup_error']=str(e)
        if not token:
            HF_STATUS['reason']='No writable HF_TOKEN in runtime; adapter is exported locally'
        else:
            from huggingface_hub import HfApi
            api=HfApi(token=token)
            api.create_repo(repo_id=HF_REPO_ID,repo_type='model',exist_ok=True)
            api.upload_folder(folder_path=str(HF_BUNDLE),repo_id=HF_REPO_ID,
                repo_type='model',commit_message='V5.6 four-way research adapter with validation selection and provenance')
            HF_STATUS.update(uploaded=True,url='https://huggingface.co/'+HF_REPO_ID)
except Exception as e:
    HF_STATUS.update(uploaded=False,error=repr(e))
    print('HF upload unavailable, preserving local checkpoint:',repr(e))
finally:
    (OUT/'hf_upload_status.json').write_text(json.dumps(HF_STATUS,indent=2))
print('HF upload status:',json.dumps(HF_STATUS,indent=2))

## 10 · Automatic complete ZIP download
The final archive includes `training.log`, `training_steps.jsonl`, `cumulative_layer_results.csv`, `four_way_validation.csv`, `four_way_diagnostic_audit.csv`, `validation_selection.json`, `fast_eval.json`, `final_audit.json`, all original candidate checkpoints and selected research-adapter files.


In [ ]:
# @title Regenerate the complete research archive and download
for item in sorted(OUT.rglob('*')):
    if item.is_file() and item.name not in ('artifact_manifest.json',):
        pass
manifest_items=[]
for item in sorted(OUT.rglob('*')):
    if item.is_file() and item.name!='artifact_manifest.json':
        manifest_items.append(dict(path=str(item.relative_to(OUT)),bytes=item.stat().st_size,
            sha256=hashlib.sha256(item.read_bytes()).hexdigest()))
(OUT/'artifact_manifest.json').write_text(json.dumps(manifest_items,indent=2))
ARCHIVE=str(OUT.parent/(OUT.name+'.zip'))
with zipfile.ZipFile(ARCHIVE,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as zipout:
    for item in sorted(OUT.rglob('*')):
        if item.is_file():zipout.write(item,str(item.relative_to(OUT)))
print('COMPLETE DOWNLOAD:',ARCHIVE,'size',Path(ARCHIVE).stat().st_size)
if AUTO_DOWNLOAD_RESULTS:
    try:
        from google.colab import files
        files.download(ARCHIVE)
    except Exception as e:
        print('Browser download unavailable:',repr(e),'Use Colab Files sidebar:',ARCHIVE)
from IPython.display import display,FileLink
display(FileLink(ARCHIVE))

### Scientific interpretation and recommended follow-up

- Arm **B** isolates the 18 chunk-cell Delta-CeNN layers from full-attention quality loss. Compare both **agreement and latency** to native A; the chunk-cell implementation reuses the Qwen gated-delta recurrence and is **not** an independently faster CeNN kernel.
- Arm **C** shows whether individually qualified V5.3 full-attention replacements add value over B without increasing NLL or slowing runtime beyond the predeclared budget.
- Arm **D** installs all six full-attention replacements temporarily **for research only**. It is never eligible for model selection or Hugging Face quality-only promotion.
- Do not claim success because 24/24 candidates trained. Report selected layers and both validation and diagnostic performance.
- `quick` is a functional experiment, not publication-level evidence. For stronger evidence run `research`, multiple seeds, longer-context, an external test corpus, and a hardware/kernel-level benchmark with optimized Qwen backends.

[Repository](https://github.com/vtavakkoli/TinyCeNN-LM) · [Official Colab](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Qwen35_08B_DeltaCeNN_V56_FourWay_QualityGated_Colab.ipynb).
